# Track 2 — Intelligent Well Archive

Optional **local inspection and keyword-retrieval baseline**, not a finished question-answering system. PDF parsing, evidence-backed answers and a frozen inference entry point are participant responsibilities. Adjust the example dataset path only after the approved laptop layout is confirmed.

In [ ]:
from pathlib import Path
import csv
import re
from html import unescape

DATA_ROOT = Path("../../data/track_2")
MANIFEST = DATA_ROOT / "metadata" / "document_manifest.csv"
DOCUMENTS = DATA_ROOT / "documents"
EXPECTED_COLUMNS = ["question_id", "answer", "evidence", "confidence"]
print("Dataset found:", DATA_ROOT.exists())
print("Manifest found:", MANIFEST.exists())
print("Documents folder found:", DOCUMENTS.exists())

In [ ]:
if MANIFEST.is_file():
    with MANIFEST.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        print("Manifest columns:", reader.fieldnames)
        print("Sample manifest rows:", [row for _, row in zip(range(3), reader)])
else:
    print("Manifest unavailable at placeholder path; confirm approved data root with technical support.")

In [ ]:
def read_local_text(path):
    if path.suffix.lower() not in {".txt", ".html", ".htm", ".xml"}:
        raise ValueError("This basic reader supports only text-like files; add a PDF parser separately.")
    raw = path.read_text(encoding="utf-8", errors="replace")
    if path.suffix.lower() in {".html", ".htm", ".xml"}:
        raw = unescape(re.sub(r"<[^>]+>", " ", raw))
    return raw

files = sorted(
    p for p in DOCUMENTS.rglob("*")
    if p.is_file() and p.suffix.lower() in {".txt", ".html", ".htm", ".xml"}
) if DOCUMENTS.is_dir() else []
print("Readable text-like files:", len(files))
if files:
    print("Preview from", files[0].relative_to(DATA_ROOT), ":", read_local_text(files[0])[:350])
print("PDF content is not covered by this baseline; use an approved local PDF parser.")

In [ ]:
def keyword_search(question, paths, top_k=3):
    stopwords = {"the", "a", "an", "in", "on", "of", "is", "what", "when", "which"}
    terms = set(re.findall(r"\w+", question.lower())) - stopwords
    matches = []
    for path in paths:
        content = read_local_text(path)
        score = sum(len(re.findall(r"\b" + re.escape(term) + r"\b", content.lower())) for term in terms)
        if score:
            matches.append((score, str(path.relative_to(DATA_ROOT)), content[:250]))
    return sorted(matches, key=lambda x: (-x[0], x[1]))[:top_k]

print(keyword_search("well drilling report", files))

## Validate your own answers CSV

The example search results are **not answers** and must not be treated as verified evidence. Use the approved question template and your own implemented QA workflow to generate predictions.

In [ ]:
PREDICTIONS = Path("outputs/track2_predictions.csv")  # set after implementing your solution
if PREDICTIONS.is_file():
    with PREDICTIONS.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)
        print("Exact output header:", reader.fieldnames == EXPECTED_COLUMNS)
        print("Rows:", len(rows))
        print("Duplicate question IDs:", len(rows) - len(set(row.get("question_id") for row in rows)))
        print("Answers missing evidence:", sum(bool(row.get("answer", "").strip()) and not row.get("evidence", "").strip() for row in rows))
else:
    print("Validator skipped: generate your own answers CSV first.")

## Your implementation

Add approved PDF parsing, retrieval or extraction, grounded answering, evidence paths, abstention and confidence. Save any fitted indexes/models before submission. Hidden evaluation may ask unseen questions against the approved archive, without rebuilding learned artefacts.